# Exploratory Data Analysis

In [33]:
import glob
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
# pip install pyarrow

## Objective: 10-Day Return Prediction

- Goal: Train a machine learning model to predict whether a stock will increase by at least 1% over the next 10 trading days, using information from the S&P 500 and its constituent companies.
- Description: The model leverages technical indicators, price action, volume dynamics, volatility measures, and lagged features to identify short-term market patterns that precede positive price movements over a 10-day horizon. The task is framed as a binary classification problem, where the target variable indicates whether the cumulative return over the next 10 days exceeds the 1% threshold.
- Purpose: To evaluate the feasibility of short-term return prediction using historical market data while strictly avoiding look-ahead bias and preserving the temporal structure of financial time series.

### Data Origin and Description

The data for this project comes from Kaggle, specifically the Advanced Stock Dataset by baidalinadilzhan, which is built using historical data retrieved from the Yahoo Finance API and covers approximately the past five years of trading for companies in the S&P 500 index. The dataset is structured with daily adjusted prices that account for corporate actions such as dividends and stock splits, making it suitable for financial analysis and time-series modeling.

The full dataset includes over 620,000 daily observations with 73 engineered features, including opening, high, low, and closing prices, trading volume, technical indicators such as moving averages and RSI, volatility measures, and multiple lagged variables. These features provide a rich foundation for short-term multi-day return forecasting, enabling the construction of a binary classification target based on the cumulative return over the next 10 trading days, which indicates whether a stock achieves a return of at least 1 percent over that horizon. All targets are derived using strictly forward-looking returns, ensuring the preservation of temporal structure and the avoidance of look-ahead bias.

### Dataset Columns
- DATE: Trading date.
- TICKER: Stock ticker symbol.
- OPEN: Opening price of the day.
- HIGH: Highest price during the day.
- LOW: Lowest price during the day.
- CLOSE: Closing price of the day.
- VOLUME: Total number of shares traded.
- DIVIDENDS: Dividend payments on that date (if any).
- STOCK_SPLITS: Stock split events on that date (if any).
- SMA_5: 5-day simple moving average of the closing price.
- SMA_10: 10-day simple moving average.
- SMA_20: 20-day simple moving average.
- SMA_50: 50-day simple moving average.
- EMA_12: 12-day exponential moving average.
- EMA_26: 26-day exponential moving average.
- MACD: Difference between EMA_12 and EMA_26 (momentum indicator).
- MACD_SIGNAL: Moving average of the MACD.
- MACD_HISTOGRAM: Difference between MACD and its signal line.
- RSI: Relative Strength Index (14 periods).
- VOLATILITY: Historical price volatility.
- BB_MIDDLE: Middle Bollinger Band (moving average).
- BB_UPPER: Upper Bollinger Band.
- BB_LOWER: Lower Bollinger Band.
- BB_WIDTH: Bollinger Band width (relative volatility measure).
- BB_POSITION: Price position within Bollinger Bands.
- PRICE_CHANGE: Daily closing price change.
- PRICE_CHANGE_5D: Cumulative price change over the last 5 days.
- HIGH_LOW_RATIO: Ratio between high and low prices.
- OPEN_CLOSE_RATIO: Ratio between open and close prices.
- VOLUME_SMA: Moving average of volume.
- VOLUME_RATIO: Current volume relative to its moving average.
- CLOSE_LAG_1: Closing price 1 day ago.
- CLOSE_LAG_2: Closing price 2 days ago.
- CLOSE_LAG_3: Closing price 3 days ago.
- CLOSE_LAG_4: Closing price 4 days ago.
- CLOSE_LAG_5: Closing price 5 days ago.
- CLOSE_LAG_6: Closing price 6 days ago.
- CLOSE_LAG_7: Closing price 7 days ago.
- CLOSE_LAG_8: Closing price 8 days ago.
- CLOSE_LAG_9: Closing price 9 days ago.
- CLOSE_LAG_10: Closing price 10 days ago.
- VOLUME_LAG_1: Volume 1 day ago.
- VOLUME_LAG_2: Volume 2 days ago.
- VOLUME_LAG_3: Volume 3 days ago.
- VOLUME_LAG_4: Volume 4 days ago.
- VOLUME_LAG_5: Volume 5 days ago.
- VOLUME_LAG_6: Volume 6 days ago.
- VOLUME_LAG_7: Volume 7 days ago.
- VOLUME_LAG_8: Volume 8 days ago.
- VOLUME_LAG_9: Volume 9 days ago.
- VOLUME_LAG_10: Volume 10 days ago.
- PRICE_CHANGE_LAG_1: Price change 1 day ago.
- PRICE_CHANGE_LAG_2: Price change 2 days ago.
- PRICE_CHANGE_LAG_3: Price change 3 days ago.
- PRICE_CHANGE_LAG_4: Price change 4 days ago.
- PRICE_CHANGE_LAG_5: Price change 5 days ago.
- PRICE_CHANGE_LAG_6: Price change 6 days ago.
- PRICE_CHANGE_LAG_7: Price change 7 days ago.
- PRICE_CHANGE_LAG_8: Price change 8 days ago.
- PRICE_CHANGE_LAG_9: Price change 9 days ago.
- PRICE_CHANGE_LAG_10: Price change 10 days ago.
- RSI_LAG_1: RSI 1 day ago.
- RSI_LAG_2: RSI 2 days ago.
- RSI_LAG_3: RSI 3 days ago.
- RSI_LAG_4: RSI 4 days ago.
- RSI_LAG_5: RSI 5 days ago.
- MACD_LAG_1: MACD 1 day ago.
- MACD_LAG_2: MACD 2 days ago.
- MACD_LAG_3: MACD 3 days ago.
- MACD_LAG_4: MACD 4 days ago.
- MACD_LAG_5: MACD 5 days ago.
- VOLATILITY_LAG_1: Volatility 1 day ago.
- VOLATILITY_LAG_2: Volatility 2 days ago.
- FUTURE_RETURN_1D / 5D / 10D / 20D: Future return over N days.
- FUTURE_UP_1D / 5D / 10D / 20D: Binary label (1 if return > 0, else 0).
- FUTURE_CATEGORY_1D / 5D / 10D / 20D: Categorical future return class.

## Load data and create DataFrame

In [34]:
# Define the path where the files are located
path = r'../../data/raw'

In [35]:
# List Parquet files starting with 'data'
all_files = glob.glob(os.path.join(path, 'data*.parquet'))

In [36]:
# Read each Parquet file and store it in a list of DataFrames
df_list = []
for filename in all_files:
    df_part = pd.read_parquet(filename, engine='pyarrow')
    df_list.append(df_part)

In [37]:
# Concatenate all DataFrames into a single DataFrame
df = pd.concat(df_list, axis=0, ignore_index=True)

In [40]:
# Convert Date to datetime (required for temporal split)
df['Date'] = pd.to_datetime(df['Date'], utc=True)
df = df.sort_values('Date')


## Descriptive Analysis

In [41]:
# Show first rows of the dataset
df.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker,SMA_5,...,Future_Category_1d,Future_Return_5d,Future_Up_5d,Future_Category_5d,Future_Return_10d,Future_Up_10d,Future_Category_10d,Future_Return_20d,Future_Up_20d,Future_Category_20d
65832,2020-07-15 04:00:00+00:00,70.223918,70.554182,64.892485,65.600197,4454200,0.0,0.0,BALL,67.734685,...,1.0,0.052647,1,3.0,0.061134,1,3.0,0.069908,1,3.0
169820,2020-07-15 04:00:00+00:00,136.696600,137.079993,133.661381,134.188553,1797100,0.0,0.0,CCI,135.372302,...,1.0,0.000357,1,2.0,0.034941,1,3.0,-0.016667,0,1.0
4942,2020-07-15 04:00:00+00:00,88.109116,89.239187,87.989673,88.871689,6205000,0.0,0.0,ABT,86.412703,...,1.0,0.035770,1,3.0,0.059857,1,3.0,0.040525,1,3.0
53382,2020-07-15 04:00:00+00:00,28.175123,28.983387,27.880345,28.850262,2511000,0.0,0.0,ACGL,27.157662,...,2.0,0.013184,1,2.0,0.017139,1,2.0,0.074489,1,3.0
171065,2020-07-15 04:00:00+00:00,80.500000,82.066002,79.260002,81.750000,883400,0.0,0.0,DAY,81.694000,...,1.0,-0.012722,0,1.0,-0.000734,0,1.0,-0.144954,0,0.0


In [42]:
# Check number of rows and columns
df.shape

(620095, 73)

In [43]:
# Display the column names of the DataFrame
df.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits', 'Ticker', 'SMA_5', 'SMA_10', 'SMA_20', 'SMA_50',
       'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'RSI',
       'BB_Middle', 'BB_Upper', 'BB_Lower', 'BB_Width', 'BB_Position',
       'Volatility', 'Price_Change', 'Price_Change_5d', 'High_Low_Ratio',
       'Open_Close_Ratio', 'Volume_SMA', 'Volume_Ratio', 'Close_lag_1',
       'Close_lag_2', 'Close_lag_3', 'Close_lag_5', 'Close_lag_10',
       'Volume_lag_1', 'Volume_lag_2', 'Volume_lag_3', 'Volume_lag_5',
       'Volume_lag_10', 'Price_Change_lag_1', 'Price_Change_lag_2',
       'Price_Change_lag_3', 'Price_Change_lag_5', 'Price_Change_lag_10',
       'RSI_lag_1', 'RSI_lag_2', 'RSI_lag_3', 'RSI_lag_5', 'RSI_lag_10',
       'MACD_lag_1', 'MACD_lag_2', 'MACD_lag_3', 'MACD_lag_5', 'MACD_lag_10',
       'Volatility_lag_1', 'Volatility_lag_2', 'Volatility_lag_3',
       'Volatility_lag_5', 'Volatility_lag_10', 'Future_Return_1d',

In [44]:
# Descriptive statistics for numerical variables
df.describe()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,SMA_5,SMA_10,SMA_20,...,Future_Category_1d,Future_Return_5d,Future_Up_5d,Future_Category_5d,Future_Return_10d,Future_Up_10d,Future_Category_10d,Future_Return_20d,Future_Up_20d,Future_Category_20d
count,620095.000000,620095.000000,620095.000000,620095.000000,6.200950e+05,620095.000000,620095.000000,620095.000000,620095.000000,620095.000000,...,620095.000000,620095.000000,620095.000000,620095.000000,620095.000000,620095.000000,620095.000000,620095.000000,620095.000000,620095.000000
mean,170.627414,172.655990,168.572121,170.644869,6.096960e+06,0.008466,0.000509,170.468481,170.252508,170.507352,...,1.529230,0.003469,0.537629,1.601787,0.006701,0.544675,1.638134,0.013391,0.546842,1.671147
std,348.798425,352.931455,344.794251,348.874888,2.343659e+07,0.150236,0.088409,348.389490,347.815938,348.346822,...,0.833302,0.045223,0.498582,1.171758,0.063206,0.498001,1.270877,0.089116,0.497801,1.341019
min,2.216000,2.318000,2.195000,2.225000,0.000000e+00,0.000000,0.000000,2.282600,2.405400,2.571350,...,0.000000,-0.587245,0.000000,0.000000,-0.605622,0.000000,0.000000,-0.641508,0.000000,0.000000
25%,54.084445,54.752876,53.418879,54.089727,1.006000e+06,0.000000,0.000000,54.064695,54.001021,54.123933,...,1.000000,-0.020331,0.000000,0.000000,-0.028207,0.000000,0.000000,-0.038917,0.000000,0.000000
50%,100.756525,102.024624,99.501965,100.802925,2.122600e+06,0.000000,0.000000,100.693167,100.565018,100.708299,...,2.000000,0.003506,1.000000,2.000000,0.006120,1.000000,2.000000,0.010784,1.000000,2.000000
75%,193.751534,196.013628,191.432234,193.721542,4.888700e+06,0.000000,0.000000,193.624097,193.474001,193.834851,...,2.000000,0.026855,1.000000,3.000000,0.040424,1.000000,3.000000,0.061390,1.000000,3.000000
max,9914.169922,9964.769531,9794.000000,9924.400391,1.543911e+09,75.000000,50.000000,9814.860156,9693.651074,9672.676465,...,3.000000,0.784177,1.000000,3.000000,1.260495,1.000000,3.000000,2.223735,1.000000,3.000000


In [45]:
# General DataFrame info
df.info()

<class 'pandas.DataFrame'>
Index: 620095 entries, 65832 to 620094
Data columns (total 73 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   Date                 620095 non-null  datetime64[us, UTC]
 1   Open                 620095 non-null  float64            
 2   High                 620095 non-null  float64            
 3   Low                  620095 non-null  float64            
 4   Close                620095 non-null  float64            
 5   Volume               620095 non-null  int64              
 6   Dividends            620095 non-null  float64            
 7   Stock Splits         620095 non-null  float64            
 8   Ticker               620095 non-null  str                
 9   SMA_5                620095 non-null  float64            
 10  SMA_10               620095 non-null  float64            
 11  SMA_20               620095 non-null  float64            
 12  SMA_50        

In [46]:
# Count missing values per column
df.isnull().sum().sort_values(ascending=False)

Date                   0
Open                   0
High                   0
Low                    0
Close                  0
                      ..
Future_Up_10d          0
Future_Category_10d    0
Future_Return_20d      0
Future_Up_20d          0
Future_Category_20d    0
Length: 73, dtype: int64

### Observations
- The dataset contains a total of 620,095 rows.
- It has 73 columns in total.
- No duplicate rows were found in the dataset.
- There are no missing values in any column.
- Out of the 73 columns, 71 are numeric.
- Two columns are of object type.

## Feature Engineering

In [47]:
# Categorize RSI values into discrete strength levels
def rsi_strength(rsi):
    if rsi < 40:
        return 0   # Débil
    elif rsi <= 60:
        return 1   # Neutral
    else:
        return 2   # Fuerte

df['rsi_strength'] = df['RSI'].apply(rsi_strength)

In [48]:
# Discretize volatility into low, medium, and high regimes based on quantiles
v1 = df['Volatility'].quantile(0.33)
v2 = df['Volatility'].quantile(0.66)

def vol_level(v):
    if v < v1:
        return 0   # Baja
    elif v < v2:
        return 1   # Media
    else:
        return 2   # Alta

df['vol_level'] = df['Volatility'].apply(vol_level)

In [49]:
# Define a selective binary target based on extreme 10-day future returns
df['target'] = np.nan

df.loc[df['Future_Return_10d'] >= 0.01, 'target'] = 1
df.loc[df['Future_Return_10d'] <= 0.0, 'target'] = 0

df = df.dropna(subset=['target'])

In [50]:
# Measure relative price position with respect to the 20-day moving average
df['price_vs_sma20'] = (df['Close'] / df['SMA_20']) - 1

In [51]:
# Select final modeling features and remove unused columns
cols_to_keep = ['Date', 'Price_Change_5d', 'price_vs_sma20', 'rsi_strength', 'Volume_Ratio', 'vol_level', 'target']

existing_cols = [c for c in cols_to_keep if c in df.columns]
df = df[existing_cols]

In [52]:
# Show final columns
df.columns

Index(['Date', 'Price_Change_5d', 'price_vs_sma20', 'rsi_strength',
       'Volume_Ratio', 'vol_level', 'target'],
      dtype='str')

### Observations

This feature engineering step is designed to keep the model simple and focused, using a small set of financially meaningful variables that provide clear trading signals. Raw indicators are transformed into robust, interpretable features to reduce noise and complexity, while preserving the most relevant information. This streamlined feature set improves model stability, interpretability, and generalization, helping the model concentrate on signals that truly matter.

## Split Temporal

In [53]:
# Sort by date
df = df.sort_values('Date').reset_index(drop=True)

In [54]:
# Define the cutoff date (oldest 80 percent for training)
split_index = int(len(df) * 0.8)

In [55]:
# Create temporal datasets
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

In [56]:
# Define X and y, and drop Date (not used in the model)
X_train = train_df.drop(columns=['target', 'Date'])
y_train = train_df['target']

X_test = test_df.drop(columns=['target', 'Date'])
y_test = test_df['target']

## Standard Scaler

In [57]:
# Standard Scaling for all features (X)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

## Save Train and Test data

In [ ]:
# Guardar los conjuntos de prueba y entrenamiento procesados como archivos CSV
X_train_scaled.to_csv('../../data/processed/X_train_eda9.csv', index=False)
X_test_scaled.to_csv('../../data/processed/X_test_eda9.csv', index=False)
y_train.to_csv('../../data/processed/y_train_eda9.csv', index=False)
y_test.to_csv('../../data/processed/y_test_eda9.csv', index=False)

OSError: Cannot save file into a non-existent directory: '..\data\processed'